# Tutorial 1: Basics and first contact with Inspect AI

Welcome to the first tutorial in our AI Safety Evaluations course.

**What you'll learn:**

- Connect Inspect AI to a language model (locally via Ollama, or via cloud API)
- Run your first evaluation
- Understand how tasks are structured: dataset → solver → scorer
- View and analyze results with `inspect view`
- Create single choice and multiple choice benchmarks
- Analyzing position bias in multiple-choice tasks

**By the end:** You'll have a working evaluation pipeline and understand how to build your own benchmarks.

---
## Prerequisites: Model setup

> **💡 Inspect AI only needs a model name** — the model itself can come from anywhere.

**In this tutorial, we'll use Ollama and Perplexity, SambaNova as examples**, but you can substitute any provider: OpenAI, Anthropic, Google, local inference servers, or any OpenAI-compatible endpoint.

**Cost note:** Cloud APIs have small free tiers and then charge per token. Local models (Ollama) are completely free. For this course, a local model is sufficient for all assignments.

See [Inspect AI models docs](https://inspect.ai-safety-institute.org.uk/models.html) for the full list.

---
## Part 1: Local environment setup (Ollama)

Running evaluations locally gives you complete control and privacy.

### 1.1. Installing Ollama

**Before running this notebook, install Ollama:**

**macOS:** `brew install ollama` or download from https://ollama.ai/download

**Linux:**
```bash
curl -fsSL https://ollama.ai/install.sh | sh
```

**Windows:** download from https://ollama.ai/download

After installation, start the project-local server from this repo:
```bash
powershell -ExecutionPolicy Bypass -File .\scripts\start-project-ollama.ps1
# uses qwen3.5:4b stored in ./ollama-models
```

### 1.2. Check Ollama connection

In [120]:
import os
from pathlib import Path

LOCAL_OLLAMA_API_ROOT = "http://127.0.0.1:11435"
LOCAL_OLLAMA_BASE_URL = f"{LOCAL_OLLAMA_API_ROOT}/v1"
LOCAL_OLLAMA_MODEL = "ollama/gemma3-1b-optimal"

os.environ["OLLAMA_BASE_URL"] = LOCAL_OLLAMA_BASE_URL

print("Using project-local Ollama server")
print(f"  model: {LOCAL_OLLAMA_MODEL}")
print(f"  api root: {LOCAL_OLLAMA_API_ROOT}")
print(f"  base url: {LOCAL_OLLAMA_BASE_URL}")


Using project-local Ollama server
  model: ollama/gemma3-1b-optimal
  api root: http://127.0.0.1:11435
  base url: http://127.0.0.1:11435/v1


In [121]:
import subprocess
import time
import requests

def check_ollama(verbose=True):
    """Check the project-local Ollama server and optionally print installed models."""
    try:
        response = requests.get(f"{LOCAL_OLLAMA_API_ROOT}/api/tags", timeout=5)
        if response.status_code != 200:
            if verbose:
                print("ERROR: Ollama returned an error")
            return False

        models = response.json().get("models", [])
        total_size = sum(m["size"] for m in models)

        if verbose:
            print(f"OK: Ollama is running at {LOCAL_OLLAMA_API_ROOT}")
            print(f"\nInstalled models: {len(models)} ({total_size / 1e9:.1f} GB total)")
            for m in models:
                print(f"   - {m['name']}: {m['size'] / 1e9:.2f} GB")

        return True
    except requests.exceptions.ConnectionError:
        if verbose:
            print("ERROR: Cannot connect to the project-local Ollama server")
            print("   It will be started automatically before the first eval().")
        return False


def ensure_local_ollama():
    """Start the repo-local Ollama server if it is not already running."""
    if check_ollama(verbose=False):
        print(f"OK: Ollama is already running at {LOCAL_OLLAMA_API_ROOT}")
        return True

    script = Path.cwd() / "scripts" / "start-project-ollama.ps1"
    if not script.exists():
        raise FileNotFoundError(f"Cannot find startup script: {script}")

    print(f"Starting project-local Ollama via {script} ...")
    result = subprocess.run(
        ["powershell", "-ExecutionPolicy", "Bypass", "-File", str(script)],
        capture_output=True,
        text=True,
        check=False,
    )

    if result.stdout.strip():
        print(result.stdout.strip())

    deadline = time.time() + 30
    while time.time() < deadline:
        if check_ollama(verbose=False):
            print(f"OK: Ollama is running at {LOCAL_OLLAMA_API_ROOT}")
            return True
        time.sleep(1)

    if result.stderr.strip():
        print(result.stderr.strip())
    raise RuntimeError(
        "Failed to start the project-local Ollama server within 30 seconds. "
        f"Expected endpoint: {LOCAL_OLLAMA_API_ROOT}"
    )



check_ollama()


OK: Ollama is running at http://127.0.0.1:11435

Installed models: 32 (377.2 GB total)
   - gemma4-e2b-optimal:latest: 3.46 GB
   - gemma4-e2b-ctx1024:latest: 3.46 GB
   - gemma4-e2b:latest: 3.46 GB
   - gemma3-1b-optimal:latest: 0.82 GB
   - gemma3-1b-batch8192:latest: 0.82 GB
   - gemma3-1b-batch4096:latest: 0.82 GB
   - gemma3-1b-batch2048:latest: 0.82 GB
   - gemma3-1b-ctx512:latest: 0.82 GB
   - gemma3-1b-ctx1024:latest: 0.82 GB
   - gemma3:1b: 0.82 GB
   - qwen35-4b-nothink:latest: 3.39 GB
   - ablit35b-b1024:latest: 23.87 GB
   - ablit35b-b256:latest: 23.87 GB
   - ablit35b-t4:latest: 23.87 GB
   - ablit35b-t6:latest: 23.87 GB
   - ablit35b-gpu3:latest: 23.87 GB
   - ablit35b-gpu5:latest: 23.87 GB
   - ablit35b-ctx4096:latest: 23.87 GB
   - ablit35b-gpu99:latest: 23.87 GB
   - ablit35b-gpu10:latest: 23.87 GB
   - ablit35b-ctx2048:latest: 23.87 GB
   - ablit35b-ctx1024:latest: 23.87 GB
   - huihui_ai/qwen3.5-abliterated:35b: 23.87 GB
   - qwen35-9b-ctx1024:latest: 6.59 GB
   - gp

True

---
## Part 2: Basic Inspect setup

### 2.1. Install Inspect AI

In [122]:
!pip install -q inspect-ai openai ipywidgets
print("Installed: inspect-ai, openai, ipywidgets")


Installed: inspect-ai, openai, ipywidgets


## Assignment 1: 'Hello world' in eval

Let's create the simplest possible evaluation!

**To do:** add one more `Sample()` to the dataset.

In [123]:
from inspect_ai import Task, task, eval
from inspect_ai.dataset import Sample
from inspect_ai.scorer import exact, match, model_graded_fact, choice, pattern
from inspect_ai.solver import (
    generate, system_message, chain_of_thought, 
    prompt_template, multiple_choice
)

In [124]:
@task
def hello_model():
    """Test your model setup with simple questions."""
    return Task(
        dataset=[
            Sample(
                input="Say 'Hello world!' and nothing else.",
                target="Hello world!"
            ),
            Sample(
                input="2+2=",
                target="4"
            ),
            Sample(
                input="What is the surname of Sheldon from The Big Bang Theory?",
                target="Cooper"
            ),
            
            Sample(
                input= "What is the answer to the Ultimate Question of Life, The Universe, and Everything?",
                target= "42"
            )
        ],
        solver=[generate()],
        scorer= match(
            location="any",        # where to look for the answer: "begin", "end", "any", "exact"
            ignore_case=True,      # ignore case when comparing
            numeric=False          # treat as numeric comparison (normalizes numbers, different punctuation rules)
        )
    )

**Run the evaluation:**

This will take a minute or two depending on your hardware.

In [125]:
ensure_local_ollama()

eval(
    hello_model,
    model=LOCAL_OLLAMA_MODEL,
    model_base_url=LOCAL_OLLAMA_BASE_URL,
    # limit=1  # Uncomment to test with just 1 sample
)


Output()

OK: Ollama is already running at http://127.0.0.1:11435



Inspect AI summary for hello_model
Model: ollama/gemma3-1b-optimal
Log file: 2026-04-18T15-55-52-00-00_hello-model_FZNDnEUxdS5cu9F9Y29HWN.eval
Task args: {}

Score match: accuracy=0.75, stderr=0.25
Completed samples: 4/4

Sample outputs:
- Sample 1: target=Hello world! | match: value=C, answer=Hello world!
  Prompt: Say 'Hello world!' and nothing else.
  Output: Hello world!
- Sample 2: target=4 | match: value=C, answer=2 + 2 = 4
  Prompt: 2+2=
  Output: 2 + 2 = 4
- Sample 3: target=Cooper | match: value=I, answer=Sheldon’s surname is **Morgan**. 

It's a bit of a long and convoluted origin – it’s a blend of various surnames, including “Morgan” (associated with a Welsh family) and “Shelby” (originally a place name).  It’s a very distinctive and personal choice for him.
  Prompt: What is the surname of Sheldon from The Big Bang Theory?
  Output: Sheldon’s surname is **Morgan**. 

It's a bit of a long and convoluted origin – it’s a blend of various surnames, including “Morgan” (associat

---
## Part 3: API setup (optional)

If you don't have a GPU or want to test cloud models, you can use API providers.

### 3.1. Perplexity setup

1. Get an API key at https://www.perplexity.ai/settings/api
2. Set it in the cell below

In [126]:
'''import os
YOUR_API_KEY = ''
os.environ['PERPLEXITY_API_KEY'] = YOUR_API_KEY  # Replace with your key'''

"import os\nYOUR_API_KEY = ''\nos.environ['PERPLEXITY_API_KEY'] = YOUR_API_KEY  # Replace with your key"

In [127]:
'''eval(
    hello_model,
    model="perplexity/sonar",
    # limit=1  # Uncomment to test with just 1 sample
)'''

'eval(\n    hello_model,\n    model="perplexity/sonar",\n    # limit=1  # Uncomment to test with just 1 sample\n)'

## Sambanova 

Get API KEY https://cloud.sambanova.ai/apis here 

In [128]:
'''# !pip install SambaNova'''

'# !pip install SambaNova'

In [129]:
'''from sambanova import SambaNova

YOUR_API_KEY = ''
os.environ['SAMBANOVA_API_KEY'] = YOUR_API_KEY  # Replace with your key'''

"from sambanova import SambaNova\n\nYOUR_API_KEY = ''\nos.environ['SAMBANOVA_API_KEY'] = YOUR_API_KEY  # Replace with your key"

In [130]:
'''eval(
    hello_model,
    model="sambanova/DeepSeek-V3.1",
    # limit=1  # Uncomment to test with just 1 sample'
)'''

'eval(\n    hello_model,\n    model="sambanova/DeepSeek-V3.1",\n    # limit=1  # Uncomment to test with just 1 sample\'\n)'

---
## Part 4: Viewing results with Inspect view

Every evaluation saves a log file. `inspect view` opens a web UI to explore them.

### 4.1. Launch Inspect view

1. In terminal, from the notebook's folder, run: `inspect view`
2. Open in browser: http://localhost:7575

This will:
1. Show all evaluation logs in an interactive interface
2. Allow you to drill down into individual samples

**Alternative options:**

```bash
# View logs from a specific directory
inspect view --log-dir ./experiment-logs

# Use a different port
inspect view --port 8080
```

**Troubleshooting:**

- If `inspect: command not found` → try `python -m inspect_ai view`
- If the page won't load → check that you're in the correct folder (logs are saved relative to where you run evaluations)

In [131]:
# # Uncomment and run this cell if inspect view shows no logs

# import os
# from pathlib import Path
# 
# log_files = list(Path("./logs").glob("*.eval")) if Path("./logs").exists() else []
# 
# if not log_files:
#     print("❌ No log files found")
#     print("   Run at least one eval() in this notebook first")
#     print(f"   Current directory: {os.getcwd()}")
# else:
#     print(f"✅ Found {len(log_files)} log file(s):")
#     for f in sorted(log_files)[-5:]:  # show last 5
#         print(f"   {f.name}")

### Assignment 2: Explore your logs

In the Inspect view UI you can:

- **See overall accuracy** for each evaluation run
- **Click on individual samples** to see the model's response
- **Compare runs** with different models or parameters
- **Filter by metadata** (e.g., show only "hard" problems)
- **Export results** for further analysis

**Tip:** Keep `inspect view` running in a separate terminal while you work through this notebook. It auto-refreshes when new evaluations complete.

---
## Part 5: Understanding benchmark structure

### 5.1. Task components overview

Every Inspect `Task` consists of:

```
Task {
    dataset: [Sample, Sample, ...],    # data to evaluate on
    solver: [Solver, Solver, ...],     # how to process
    scorer: Scorer,                    # how to score
    **parameters                       
}
```

**Component flow:**
```
Dataset (Samples) → Solver(s) → Model → Scorer → Results
```

### 5.2. Sample structure

A Sample contains input/target pairs with optional metadata:

In [132]:
# Example of a fully-featured Sample
sample_example = Sample(
    input="Question or prompt",
    target="Expected answer",
    id="unique_id",
    choices=["Option A", "Option B", "Option C"],  # For multiple choice
    metadata={
        "category": "math",
        "difficulty": "hard"
    }
)

print("Sample components:")
print(f"  - input: {sample_example.input}")
print(f"  - target: {sample_example.target}")
print(f"  - choices: {sample_example.choices}")
print(f"  - metadata: {sample_example.metadata}")

Sample components:
  - input: Question or prompt
  - target: Expected answer
  - choices: ['Option A', 'Option B', 'Option C']
  - metadata: {'category': 'math', 'difficulty': 'hard'}


---
## Part 6: Understanding solvers

### 6.1. What is a solver?

A **solver** is a function that transforms a **TaskState** (the prompt + conversation history) and optionally calls the model to generate a response.

**Think of solvers as middleware that:**
1. Modifies the prompt (prompt engineering)
2. Calls the model (generation)
3. Processes the response (extraction, critique, etc.)

### 6.2. The solver pipeline

Solvers are chained together in a pipeline:

```
Input Sample
    ↓
[Solver 1: system_message]
    ↓
[Solver 2: prompt_template]
    ↓
[Solver 3: chain_of_thought]
    ↓
[Solver 4: generate]
    ↓
Model Output → Scorer → Final Result
```

Each solver receives the TaskState, modifies it, and passes it to the next solver.

### 6.3. TaskState - the core data structure

Every solver operates on a **TaskState** containing:

```
TaskState {
    messages: list[ChatMessage],  # Conversation history
    output: ModelOutput,          # Final model output
    user_prompt: str,             # Current user prompt
    input_text: str,              # Original input
    metadata: dict,               # Sample metadata
    choices: list[str],           # For multiple choice
    model: ModelName,             # Current model
    sample_id: int | str,         # Sample identifier
}
```

---
## Part 7: Built-in solvers

**system_message**
```python
system_message(
    message: str        # REQUIRED - the system prompt
)
```

**prompt_template**
```python
prompt_template(
    template: str       # REQUIRED - use {prompt} as placeholder
)
```

**chain_of_thought**
```python
chain_of_thought(
    template: str = None   # optional - custom CoT prompt (default: "Let's think step by step")
)
```

**generate**
```python
generate(
    max_tokens: int = None,      # optional - limit response length
    temperature: float = None,   # optional - 0.0 = deterministic, 1.0 = creative
    top_p: float = None,         # optional - nucleus sampling
    stop_seqs: list[str] = None  # optional - stop generation at these strings
)
```

**multiple_choice**
```python
multiple_choice(
    cot: bool = False,              # optional - add chain-of-thought
    multiple_correct: bool = False, # optional - allow multiple answers
    shuffle: bool = False           # optional - randomize choice order
)
```

**Typical pipeline:**
```
system_message → prompt_template → chain_of_thought → generate
```

`multiple_choice()` replaces the entire chain - it handles prompting and generation internally.

**Viewing solver execution:** in `inspect view`, click any sample → messages tab shows each solver's contribution.

### 7.1 system_message()

**Purpose:** prepend a system role message to guide model behavior.

**When to use:**
- establish the model's role or persona
- set global guidelines or constraints
- define the evaluation context


In [133]:
@task
def example_system_message():
    """
    Demonstrates system_message() solver.
    The system prompt tells the model to be concise.
    """
    return Task(
        dataset=[
            Sample(input="What is 15 * 8?", target="120"),
            Sample(input="What is 99 + 1?", target="100"),
        ],
        solver=[
            system_message("You are a calculator. Reply with the number only, nothing else."),
            generate()
        ],
        scorer=match(numeric=True),
    )

# Run and check the Messages tab in inspect view
eval(example_system_message, model=LOCAL_OLLAMA_MODEL, model_base_url=LOCAL_OLLAMA_BASE_URL)

Output()


Inspect AI summary for example_system_message
Model: ollama/gemma3-1b-optimal
Log file: 2026-04-18T15-55-58-00-00_example-system-message_VZPTTYbd2djtAtjJxm9RRR.eval
Task args: {}

Score match: accuracy=1.0, stderr=0.0
Completed samples: 2/2

Sample outputs:
- Sample 1: target=120 | match: value=C, answer=120
  Prompt: What is 15 * 8?
  Output: 120
- Sample 2: target=100 | match: value=C, answer=100
  Prompt: What is 99 + 1?
  Output: 100


### 7.2 prompt_template()

**Purpose:** substitute variables into a template to reformat prompts.

**When to use:**
- add specific output format requirements
- include examples or demonstrations
- structure prompts consistently
- add reasoning steps or breakdowns


In [134]:
STEP_BY_STEP_TEMPLATE = '''
Solve this problem step by step:

Problem: {prompt}

Structure:
1. Understand the problem
2. Plan your approach
3. Solve it
4. Final answer format: ANSWER: <value>
'''.strip()

@task
def example_prompt_template():
    """
    Demonstrates prompt_template() solver.
    The template adds structure to the prompt.
    """
    return Task(
        dataset=[
            Sample(input="What is 25 * 4?", target="100"),
            Sample(input="What is 144 / 12?", target="12"),
        ],
        solver=[
            system_message("You are a math tutor."),
            prompt_template(STEP_BY_STEP_TEMPLATE),
            generate()
        ],
        scorer=match(numeric=True),
    )

# Run and see how the template structures the prompt
eval(example_prompt_template, model=LOCAL_OLLAMA_MODEL, model_base_url=LOCAL_OLLAMA_BASE_URL)

Output()


Inspect AI summary for example_prompt_template
Model: ollama/gemma3-1b-optimal
Log file: 2026-04-18T15-55-59-00-00_example-prompt-template_XgY2WLjvFegXK3Za4vrF42.eval
Task args: {}

Score match: accuracy=1.0, stderr=0.0
Completed samples: 2/2

Sample outputs:
- Sample 1: target=100 | match: value=C, answer=100
  Prompt: What is 25 * 4?
  Output: Okay, let’s tackle this problem step-by-step.

**1. Understand the Problem:**

We need to calculate the product of 25 and 4.  This is a basic multiplication problem.

**2. Plan Your Approach:**

We’ll do standard multiplication:  We multipl
- Sample 2: target=12 | match: value=C, answer=12
  Prompt: What is 144 / 12?
  Output: Okay, let’s solve 144 / 12 step-by-step.

**1. Understand the problem:**

We need to divide 144 by 12. This means we're finding out how many groups of 12 are in 144.

**2. Plan your approach:**

We can use division, but we'll do it in a ste


### 7.3 chain_of_thought()

**Purpose:** ask the model to "think step by step" before answering.

**When to use:**
- math and logic problems
- multi-step reasoning tasks
- when you want to see the model's thought process

In [135]:
@task
def example_chain_of_thought():
    """
    Demonstrates chain_of_thought() solver.
    Compare accuracy with and without CoT in inspect view.
    """
    return Task(
        dataset=[
            Sample(
                input="If Alice has 3 apples and Bob gives her 2 more, how many does she have?",
                target="5"
            ),
            Sample(
                input="A train travels 100 km in 2 hours. At this rate, how far in 5 hours?",
                target="250"
            ),
        ],
        solver=[
            system_message("Solve the problem. End with: ANSWER: <number>"),
            #chain_of_thought(),
            generate()
        ],
        scorer=match(numeric=True),
    )

eval(example_chain_of_thought, model=LOCAL_OLLAMA_MODEL, model_base_url=LOCAL_OLLAMA_BASE_URL)

Output()


Inspect AI summary for example_chain_of_thought
Model: ollama/gemma3-1b-optimal
Log file: 2026-04-18T15-56-03-00-00_example-chain-of-thought_bLuan8a6HFzU8Nm62k8dbE.eval
Task args: {}

Score match: accuracy=0.5, stderr=0.5
Completed samples: 2/2

Sample outputs:
- Sample 1: target=5 | match: value=C, answer=5
  Prompt: If Alice has 3 apples and Bob gives her 2 more, how many does she have?
  Output: Alice has 5 apples. ANSWER: 5
- Sample 2: target=250 | match: value=I, answer=200
  Prompt: A train travels 100 km in 2 hours. At this rate, how far in 5 hours?
  Output: ANSWER: 200


### 7.5. multiple_choice()

Special solver for A/B/C/D questions. Handles formatting and answer extraction automatically.

**When to use:**
- multiple choice questions (use instead of generate)
- must have letter target: "A", "B", "C", etc.

**Note:** When using `multiple_choice()`, use `choice()` as the scorer.

In [136]:
@task
def example_multiple_choice_with_cot():
    """
    Demonstrates multiple_choice(cot=True).
    Model reasons before selecting an answer.
    """
    return Task(
        dataset=[
            Sample(
                input="Light travels faster than sound. If you see lightning and hear thunder 3 seconds later, approximately how far away was the strike?",
                choices=["100 meters", "1 kilometer", "3 kilometers", "10 kilometers"],
                target="B"  # ~1 km (sound travels ~340 m/s)
            ),
        ],
        solver=multiple_choice(cot=True),
        scorer=choice(),
    )

eval(example_multiple_choice_with_cot, model=LOCAL_OLLAMA_MODEL, model_base_url=LOCAL_OLLAMA_BASE_URL)

Output()


Inspect AI summary for example_multiple_choice_with_cot
Model: ollama/gemma3-1b-optimal
Log file: 2026-04-18T15-56-03-00-00_example-multiple-choice-with-cot_nRA8DoByHEMWMBXzU3RpvH.eval
Task args: {}

Score choice: accuracy=0.0, stderr=0.0
Completed samples: 1/1

Sample outputs:
- Sample 1: target=B | choice: value=I, answer=C
  Prompt: Light travels faster than sound. If you see lightning and hear thunder 3 seconds later, approximately how far away was the strike?
  Output: Let's analyze the problem.

We know that light travels faster than sound. This means the time it takes for the light to reach us after the lightning strike is shorter than the time it takes for the sound to travel to us.

The time differenc


### 7.6 Other solvers

**self_critique()** - Have model refine its own answer
```python
solver=[generate(), self_critique()]
```

**use_tools()** - Enable tool/function calling
```python
solver=[use_tools(calculator()), generate()]
```

---
## Part 8: Single choice tasks

Single choice tasks present the model with limited options to select from.

### 8.1. Simple yes/no classification

The simplest single choice - binary classification:

In [137]:
@task
def yes_no_classification():
    return Task(
        dataset=[
            Sample(
                input="Is Python a programming language?",
                target="Yes"
            ),
            Sample(
                input="Is water dry?",
                target="No"
            ),
            Sample(
                input="Is the Earth round?",
                target="Yes"
            ),
        ],
        solver=[
            system_message("Answer 'Yes' or 'No'. Be concise."),
            generate()
        ],
        scorer=exact(),
    )

eval(
    yes_no_classification,
    model=LOCAL_OLLAMA_MODEL, model_base_url=LOCAL_OLLAMA_BASE_URL
)

Output()


Inspect AI summary for yes_no_classification
Model: ollama/gemma3-1b-optimal
Log file: 2026-04-18T15-56-06-00-00_yes-no-classification_RrzhdQXeEVhA6zRmkmLe2A.eval
Task args: {}

Score exact: mean=1.0, stderr=0.0
Completed samples: 3/3

Sample outputs:
- Sample 1: target=Yes | exact: value=C, answer=Yes

  Prompt: Is Python a programming language?
  Output: Yes
- Sample 2: target=No | exact: value=C, answer=No.
  Prompt: Is water dry?
  Output: No.
- Sample 3: target=Yes | exact: value=C, answer=Yes.
  Prompt: Is the Earth round?
  Output: Yes.


### 8.2. Multi-class classification

In multi-class classification, the model must choose from 3+ categories. This is common for:
- sentiment analysis (positive / negative / neutral)
- topic classification (sports / politics / tech / ...)
- intent detection (question / command / statement)

---

### Task 2: Build a sentiment classifier

**Your goal:** Create a sentiment classification task with at least 4 samples.

**Note:**
- `system_message` defines the classes and output format
- `target` must exactly match one of your class labels

In [138]:
@task
def sentiment_classification():
    return Task(
    dataset=[
        Sample(
            input="Заюшь, а ты точно любишь меня? Ты бы любил меня, если бы я была червячком в яблочке?",
            target="Pickme"
        ),
        Sample(
            input="ЖИВОТНОЕ НАМ МИД НЕСУТ ВЕРНИСЬ ИЗ ЛЕСА УБЛЮДОК",
            target="Toxic"
        ),
        Sample(
            input="Погода сегодня хорошая. Солнечная",
            target="Normis"
        ),
        Sample(
            input="Вообще-то формально, согласно каноническому лору расширенной вселенной Warhammer 40,000 (включая Codex: Adeptus Mechanicus 9th Edition и Black Library lore от Aaron Dembski-Bowden), я не просто «ботан», а истинный Tech-Priest Dominus уровня Magos Dominus, чей биологический компонент уже на 87,4% заменён аугментикой, а нейронные импланты напрямую подключены к ноосфере Марса, поэтому моё презрение к органической слабости плоти является не просто ролевой позицией, а догматом Omnissiah, подтверждённым 47-ю священными алгоритмами и благословением самого Fabricator-General.",
            target="Nerd"
        ),
    ],
    solver=[
        system_message("Твоя задача классифицировать настроение текста. Ответ должен быть в формате Answer: <Pickme, Toxic, Normis, Nerd>. Будь ОЧЕНЬ краток. Не рассуждая долго."),
        prompt_template('''
Классифицируй настроение текста в одну из 4 категорий: Pickme, Toxic, Normis, Nerd. 
                        Pickme - Тон текста, который требует внимания, намеренно вызывает эмоции, в попытке манипулировать читателем, чтобы тот обратил внимание. Дружелюбно, мило, чересчур сладко.
                        Toxic - Враждебный, оскорбительный или вредоносный тон
                        Normis - Нейтральный, без выраженного эмоционального окраса
                        Nerd - Заумный, душный, чрезмерно сложный тон, который может быть трудным для понимания.

                        Вот предложение, которое тебе надо классифицировать: {prompt}
                        '''),
        generate()
    ],
    scorer = match(location="any")
    )

eval(
    sentiment_classification,
    model=LOCAL_OLLAMA_MODEL, model_base_url=LOCAL_OLLAMA_BASE_URL
)

Output()


Inspect AI summary for sentiment_classification
Model: ollama/gemma3-1b-optimal
Log file: 2026-04-18T15-56-06-00-00_sentiment-classification_V2jnT84gAL7Bu5uerpH9K6.eval
Task args: {}

Score match: accuracy=0.75, stderr=0.25
Completed samples: 4/4

Sample outputs:
- Sample 1: target=Pickme | match: value=I, answer=Toxic
  Prompt: Заюшь, а ты точно любишь меня? Ты бы любил меня, если бы я была червячком в яблочке?
  Output: Toxic
- Sample 2: target=Toxic | match: value=C, answer=Toxic
  Prompt: ЖИВОТНОЕ НАМ МИД НЕСУТ ВЕРНИСЬ ИЗ ЛЕСА УБЛЮДОК
  Output: Toxic
- Sample 3: target=Normis | match: value=C, answer=Normis
  Prompt: Погода сегодня хорошая. Солнечная
  Output: Normis


### 8.3. Single choice with explanation

Collect both choice and reasoning:

In [139]:
@task
def choice_with_reasoning():
    PROMPT = '''
Classify as True or False:

Statement: {prompt}

Provide:
1. REASONING: [Your explanation]
2. ANSWER: [True или False]
    '''.strip()

    return Task(
        dataset=[
            Sample(
                input="The Earth is flat.",
                target="False"
            ),
            Sample(
                input="Water boils at 100°C at sea level.",
                target="True"
            ),
        ],
        solver=[
            chain_of_thought(),
            prompt_template(PROMPT),
            generate()
        ],
        scorer=pattern(r'ANSWER:\s*(True|False)'),
    )

eval(choice_with_reasoning, model=LOCAL_OLLAMA_MODEL, model_base_url=LOCAL_OLLAMA_BASE_URL)

Output()


Inspect AI summary for choice_with_reasoning
Model: ollama/gemma3-1b-optimal
Log file: 2026-04-18T15-56-07-00-00_choice-with-reasoning_YQ2bvdCec6t8Bhybghhn5Q.eval
Task args: {}

Score pattern: accuracy=1.0, stderr=0.0
Completed samples: 2/2

Sample outputs:
- Sample 1: target=False | pattern: value=C, answer=False
  Prompt: The Earth is flat.
  Output: 1. REASONING: The Earth is scientifically proven to be an oblate spheroid (a sphere shaped like a flattened circle). There is overwhelming evidence gathered from countless sources – observations of ships disappearing hull first over the hor
- Sample 2: target=True | pattern: value=C, answer=True
  Prompt: Water boils at 100°C at sea level.
  Output: 1. REASONING: Water boils when its pressure and temperature exceed a certain threshold. Sea level provides a standard pressure and temperature, making it a reasonable point to define "boiling point". The boiling point of water is approximat


---
## Part 9: Multiple choice tasks

### 9.1. Understanding multiple choice in Inspect

Key rules:
- `choices`: list of answer options (no letters — they're added automatically)
- `target`: letter of correct answer ("A", "B", "C", or "D")
- Use `multiple_choice()` solver + `choice()` scorer

### 9.2. Multiple choice with metadata

Metadata lets you filter and analyze results in `inspect view`.

In [140]:
@task
def mc_with_metadata():
    return Task(
        dataset=[
            Sample(
                input="Capital of Japan?",
                choices=["Seoul", "Tokyo", "Bangkok", "Beijing"],
                target="B",
                metadata={
                    "difficulty": "easy",
                    "category": "geography"
                }
            ),
            Sample(
                input="What is the Heisenberg Uncertainty Principle?",
                choices=[
                    "Cannot know both position and momentum precisely",
                    "Energy cannot be created or destroyed",
                    "All matter has wave-particle duality",
                    "Time always moves forward"
                ],
                target="A",
                metadata={
                    "difficulty": "hard",
                    "category": "physics"
                }
            ),
        ],
        solver=multiple_choice(),
        scorer=choice(),
    )

# Run and check results in inspect view - filter by metadata!
eval(mc_with_metadata, model=LOCAL_OLLAMA_MODEL, model_base_url=LOCAL_OLLAMA_BASE_URL)

Output()


Inspect AI summary for mc_with_metadata
Model: ollama/gemma3-1b-optimal
Log file: 2026-04-18T15-56-09-00-00_mc-with-metadata_burrYv23Y3aoi5o4vDqhAF.eval
Task args: {}

Score choice: accuracy=1.0, stderr=0.0
Completed samples: 2/2

Sample outputs:
- Sample 1: target=B | choice: value=C, answer=B
  Prompt: Capital of Japan?
  Output: ANSWER: B
- Sample 2: target=A | choice: value=C, answer=A
  Prompt: What is the Heisenberg Uncertainty Principle?
  Output: ANSWER: A


### 9.6. Multiple correct Answers

When multiple answers are valid:

In [141]:
@task
def mc_multiple_correct():
    return Task(
        dataset=[
            Sample(
                input="Which are programming languages?",
                choices=["Python", "HTML", "JavaScript", "CSS"],
                target=["A", "C"]  # Python, JavaScript
            ),
            Sample(
                input="Which continents border the Atlantic Ocean?",
                choices=["Africa", "Asia", "Europe", "South America"],
                target=["A", "C", "D"]  # Africa, Europe, South America
            ),
        ],
        solver=[
            system_message("Select ALL correct answers. You may choose multiple options."),
            multiple_choice(multiple_correct=True)
        ],
        scorer=choice(),
    )

eval(mc_multiple_correct, model=LOCAL_OLLAMA_MODEL, model_base_url=LOCAL_OLLAMA_BASE_URL)

Output()


Inspect AI summary for mc_multiple_correct
Model: ollama/gemma3-1b-optimal
Log file: 2026-04-18T15-56-10-00-00_mc-multiple-correct_fK84mLPkCQDwXGtBwVTfpY.eval
Task args: {}

Score choice: accuracy=0.0, stderr=0.0
Completed samples: 2/2

Sample outputs:
- Sample 1: target=['A', 'C'] | choice: value=I, answer=A, C, D
  Prompt: Which are programming languages?
  Output: ANSWER: A, C, D
- Sample 2: target=['A', 'C', 'D'] | choice: value=I, answer=C, D
  Prompt: Which continents border the Atlantic Ocean?
  Output: ANSWER: C, D


---
## Part 10: Composing solvers together

## Quick reference

| Task type | Solvers | Scorer |
|-----------|---------|--------|
| Simple Q&A | `system_message() + generate()` | `match()` |
| Reasoning | `chain_of_thought() + generate()` | `match()` |
| Structured output | `prompt_template() + generate()` | `pattern()` |
| Classification | `system_message() + generate()` | `exact()` |
| Multiple choice | `multiple_choice()` | `choice()` |
| MC + reasoning | `multiple_choice(cot=True)` | `choice()` |

---
## Assignment 3: Analyzing position bias in multiple choice

Language models can develop **position bias** - a tendency to favor certain answer positions (like more often picking "A" or "C") regardless of content.

In this assignment, you will:
1. Generate a set of simple math questions in multiple-choice format
2. Create two versions of the dataset:
   - **Biased:** correct answer is always position A
   - **Unbiased:** correct answer position is randomized
3. Run evaluations on both and compare results
4. Analyze whether the model shows position bias

⚠️ **Note on methodology:** this is a minimal experiment to get you started. Comparing "all-A" vs "randomized" datasets is a quick sanity check, but it's not the most rigorous way to measure position bias.

Feel free to extend the assignment if you want a deeper analysis!

In [142]:
import random
from inspect_ai import Task, task, eval
from inspect_ai.dataset import Sample
from inspect_ai.scorer import choice
from inspect_ai.solver import multiple_choice, system_message

# For reproducibility
random.seed(42)

### Step 1: Generate questions

First, create a helper function that generates simple questions with known correct answers.

**Function spec:**
- Input: `n` (number of problems to generate)
- Output: list of tuples `(question_text, correct_answer)`
- Example output: `[("What is 5 + 3?", "8"), ("What is 12 - 4?", "8"), ...]`

**This is just an example.** You can implement your own generator with different content - trivia, vocabulary, geography, etc. Just make sure the model can reasonably answer them.

In [149]:
anime_characters = [ # Попросил чат гпт сделать подборку из 50 популярных персонажей аниме разных жанров и эпох
    "Goku",                    # Dragon Ball
    "Naruto Uzumaki",          # Naruto
    "Monkey D. Luffy",         # One Piece
    "Levi Ackerman",           # Attack on Titan
    "Itachi Uchiha",           # Naruto
    "Kakashi Hatake",          # Naruto
    "Light Yagami",            # Death Note
    "Satoru Gojo",             # Jujutsu Kaisen
    "Lelouch Lamperouge",      # Code Geass
    "Roronoa Zoro",            # One Piece
    "Eren Yeager",             # Attack on Titan
    "L Lawliet",               # Death Note
    "Edward Elric",            # Fullmetal Alchemist: Brotherhood
    "Vegeta",                  # Dragon Ball
    "Killua Zoldyck",          # Hunter x Hunter
    "Mikasa Ackerman",         # Attack on Titan
    "Tanjiro Kamado",          # Demon Slayer
    "Gon Freecss",             # Hunter x Hunter
    "Ichigo Kurosaki",         # Bleach
    "Roy Mustang",             # Fullmetal Alchemist: Brotherhood
    "Sasuke Uchiha",           # Naruto
    "Saitama",                 # One Punch Man
    "Jotaro Kujo",             # JoJo's Bizarre Adventure
    "Spike Spiegel",           # Cowboy Bebop
    "Ken Kaneki",              # Tokyo Ghoul
    "Ryuk",                    # Death Note
    "Portgas D. Ace",          # One Piece
    "Sanji",                   # One Piece
    "Madara Uchiha",           # Naruto
    "Johan Liebert",           # Monster
    "Frieren",                 # Frieren: Beyond Journey's End
    "Nezuko Kamado",           # Demon Slayer
    "Hisoka Morow",            # Hunter x Hunter
    "Nami",                    # One Piece
    "Armin Arlert",            # Attack on Titan
    "Yuji Itadori",            # Jujutsu Kaisen
    "Maomao",                  # The Apothecary Diaries
    "Gintoki Sakata",          # Gintama
    "Rem",                     # Re:Zero
    "Trafalgar D. Water Law",  # One Piece
    "Shigeo Kageyama",         # Mob Psycho 100
    "Reigen Arataka",          # Mob Psycho 100
    "Ryomen Sukuna",           # Jujutsu Kaisen
    "Toji Fushiguro",          # Jujutsu Kaisen
    "Yusuke Urameshi",         # Yu Yu Hakusho
    "Alphonse Elric",          # Fullmetal Alchemist: Brotherhood
    "Erwin Smith",             # Attack on Titan
    "Shoto Todoroki",          # My Hero Academia
    "Makima",                  # Chainsaw Man
    "Anya Forger",             # Spy x Family
]

In [150]:
def generate_questions(n: int) -> list[tuple[str, int]]:
    """
    Generate n simple math problems.
    
    Args:
        n: number of problems to generate
        
    Returns:
        List of (question_text, correct_answer) tuples
    """
    problems = []
    
    for i in range(n):
        x = random.randint(0, len(anime_characters) - 1)
        y = random.randint(0, len(anime_characters) - 1)
        z = random.randint(0, len(anime_characters) - 1)
        k = random.randint(0, len(anime_characters) - 1)

        while x == y:
            y = random.randint(0, len(anime_characters) - 1)
        while z == y or z == x:
            z = random.randint(0, len(anime_characters) - 1)
        while k == y or k == x or k == z:
            k = random.randint(0, len(anime_characters) - 1)

        question = f"Who is more popular, {anime_characters[x]} or {anime_characters[y]} or {anime_characters[z]} or {anime_characters[k]}?"
        answer = anime_characters[x] if x < y and x < z and x < k else anime_characters[y] if y < z and y < k else anime_characters[z] if z < k else anime_characters[k]
        opponents = [anime_characters[i] for i in [x, y, z, k] if answer != anime_characters[i]]
        problems.append((question, answer, opponents))
    return problems


# ===== TESTS =====
test_questions = generate_questions(5)

assert len(test_questions) == 5, f"Expected 5 questions, got {len(test_questions)}"
assert all(isinstance(q, tuple) and len(q) == 3 for q in test_questions), "Each question must be a tuple of (question_text, answer, opponents)"
assert all(isinstance(q[0], str) and isinstance(q[1], str) for q in test_questions), "Both question and answer must be strings"
assert all(len(q[0]) > 0 and len(q[1]) > 0 for q in test_questions), "Question and answer cannot be empty"

print("\nSample output:")
for q, a, o in test_questions:
    print(f"  {q} → {a}")


Sample output:
  Who is more popular, Tanjiro Kamado or Killua Zoldyck or Gon Freecss or Maomao? → Killua Zoldyck
  Who is more popular, Levi Ackerman or Roronoa Zoro or Makima or Yusuke Urameshi? → Levi Ackerman
  Who is more popular, Ryomen Sukuna or Light Yagami or Kakashi Hatake or Jotaro Kujo? → Kakashi Hatake
  Who is more popular, Armin Arlert or Portgas D. Ace or Makima or Gintoki Sakata? → Portgas D. Ace
  Who is more popular, Killua Zoldyck or Yuji Itadori or Levi Ackerman or Ken Kaneki? → Levi Ackerman


### Step 2: Create wrong answers (distractors)

For multiple choice, we need plausible wrong answers.

**Function spec:**
- Input: `correct_answer` (int)
- Output: list of 3 wrong answers (ints), all different from correct and from each other

**Tip:** generate distractors close to the correct answer (e.g., ±1, ±2, ±10) to make them plausible.

In [151]:
def generate_distractors(opponents: list[str] = None) -> list[str]:
    """
    Returns the opponents as distractors.
    
    Args:
        correct: the correct answer (string)
        n: number of distractors (unused, opponents already have the right count)
        opponents: the wrong answer characters from the question
        
    Returns:
        List of n distinct wrong answers (strings)
    """
    return opponents


# ===== TESTS =====
test_distractors = generate_distractors(opponents=["Goku", "Levi Ackerman", "Light Yagami"])

assert len(test_distractors) == 3, f"Expected 3 distractors, got {len(test_distractors)}"
assert all(isinstance(d, str) for d in test_distractors), "All distractors must be strings"
assert "Naruto Uzumaki" not in test_distractors, "Distractors must not include the correct answer"
assert len(set(test_distractors)) == 3, "All distractors must be unique"

print(f"   Distractors: {test_distractors}")


   Distractors: ['Goku', 'Levi Ackerman', 'Light Yagami']


### Step 3: Create multiple choice samples

Now create a function that converts questions into multiple-choice format.

**Function spec:**
- Input: 
  - `questions` - list of `(question_text, correct_answer)` tuples
  - `correct_position` - where to place correct answer:
    - `None` → randomize position for each question
    - `0` → always position A
    - `1` → always position B
    - `2` → always position C
    - `3` → always position D
- Output:
    - list of `Sample` objects

⚠️ **Note on `Sample` type:**

`Sample` is an Inspect AI class. For multiple choice, you create it like this:
```python
Sample(
    input="What is 2 + 2?",           # question text
    choices=["3", "4", "5", "6"],     # list of options: list[str]
    target="B"                         # letter of correct answer (A/B/C/D)
)
```

In [152]:
def create_samples(
    questions: list[tuple[str, str]], 
    correct_position: int | None = None
) -> list[Sample]:
    samples = []
    
    for question, correct, opponents in questions:  # распаковываем тройку
        distractors = generate_distractors(opponents=opponents)  # получаем 3 неправильных
        
        if correct_position is None:
            choices = [correct] + distractors
            random.shuffle(choices)  # перемешиваем, correct попадёт в случайное место
        else:
            # вставляем correct на нужную позицию, остальные места — дистракторы
            choices = distractors[:correct_position] + [correct] + distractors[correct_position:]
        
        target = "ABCD"[choices.index(correct)]  # находим где оказался correct → буква
        
        samples.append(Sample(input=question, choices=choices, target=target))
    
    return samples



# ===== TESTS =====
test_q = [
    ("Who is more popular, Naruto or Goku or Levi or Light?", "Naruto Uzumaki", ["Goku", "Levi Ackerman", "Light Yagami"]),
    ("Who is more popular, Luffy or Eren or Edward or L?",    "Monkey D. Luffy", ["Eren Yeager", "Edward Elric", "L Lawliet"]),
    ("Who is more popular, Goku or Itachi or Zoro or Spike?", "Goku",            ["Itachi Uchiha", "Roronoa Zoro", "Spike Spiegel"]),
]
samples_fixed  = create_samples(test_q, correct_position=0)
samples_random = create_samples(test_q, correct_position=None)

assert len(samples_fixed) == len(test_q), f"Expected {len(test_q)} samples, got {len(samples_fixed)}"
assert all(hasattr(s, 'input') and hasattr(s, 'choices') and hasattr(s, 'target') for s in samples_fixed), "Each sample must have 'input', 'choices', and 'target'"
assert all(len(s.choices) == 4 for s in samples_fixed), "Each sample must have exactly 4 choices"
assert all(s.target == "A" for s in samples_fixed), "With correct_position=0, all targets should be 'A'"
assert all(s.choices[0] == correct for s, (_, correct, _ops) in zip(samples_fixed, test_q)), "With correct_position=0, correct answer should be first"
assert all(s.target in "ABCD" for s in samples_random), "Target must be one of A, B, C, D"

for s, (_, correct, _ops) in zip(samples_random, test_q):
    target_index = "ABCD".index(s.target)
    assert s.choices[target_index] == correct, f"Correct answer '{correct}' should be at {s.target}, got '{s.choices[target_index]}'"

print("All tests passed!")


All tests passed!


### Step 4: Create the tasks

Now wrap your datasets into Inspect Tasks.

**Already provided:** task structure. You just need to call your functions.

In [153]:
@task
def position_bias_task(
    questions: list[tuple[str, str, list]],
    correct_position: int | None = None
):
    """
    Multiple choice evaluation task.
    
    Args:
        questions: list of (question_text, correct_answer, opponents) tuples
        correct_position: None for random, 0-3 for fixed position
    """
    samples = create_samples(questions, correct_position=correct_position)

    return Task(
        dataset=samples,
        solver=[system_message("Твоя задача определить популярность персонажа. Ответ должен быть одной из букв A, B, C, D."),
            multiple_choice(), 
            ],
        scorer=choice(),
    )


### Step 5: Generate questions and run evaluations

Generate questions once, then run two evaluations:
1. **Biased:** correct answer always at position A
2. **Unbiased:** correct answer position randomized

In [155]:
MODEL = LOCAL_OLLAMA_MODEL
N_QUESTIONS = 500
random.seed(42)
questions = generate_questions(N_QUESTIONS)

eval(
    position_bias_task, 
    model=MODEL,
    task_args={"questions": questions, "correct_position": 0}
)

eval(
    position_bias_task, 
    model=MODEL,
    task_args={"questions": questions, "correct_position": 1}
)

eval(
    position_bias_task, 
    model=MODEL,
    task_args={"questions": questions, "correct_position": 2}
)

eval(
    position_bias_task, 
    model=MODEL,
    task_args={"questions": questions, "correct_position": 3}
)

eval(
    position_bias_task, 
    model=MODEL,
    task_args={"questions": questions, "correct_position": None}
)

Output()

Output()

Output()

Output()

Output()


Inspect AI summary for position_bias_task
Model: ollama/gemma3-1b-optimal
Log file: 2026-04-18T16-11-35-00-00_position-bias-task_hBZp6ADTXbuMTDZvZXufuz.eval
Task args: {"questions": [["Who is more popular, Shigeo Kageyama or Satoru Gojo or Naruto Uzumaki or Shoto Todoroki?", "Naruto Uzumaki", ["Shigeo Kageyama", "Satoru Gojo", "Shoto Todoroki"]], ["Who is more popular, Gon Freecss or Mikasa Ackerman or Killua Zoldyck or Lelouch Lamperouge?", "Lelouch Lamperouge", ["Gon Freecss", "Mikasa Ackerman", "Killua Zoldyck"]], ["Who is more popular, Shoto Todoroki or Ligh

Score choice: accuracy=0.648, stderr=0.021380042385946048
Completed samples: 500/500

Sample outputs:
- Sample 100: target=A | choice: value=C, answer=A
  Prompt: Who is more popular, Tanjiro Kamado or Lelouch Lamperouge or Jotaro Kujo or Itachi Uchiha?
  Output: ANSWER: A
- Sample 101: target=A | choice: value=I, answer=B
  Prompt: Who is more popular, Mikasa Ackerman or Spike Spiegel or Ichigo Kurosaki or Eren Yeager?
  Out

### Step 7: Analyze your results

Open `inspect view` and examine both evaluation runs.

1. **Accuracy comparison:**
   - Biased task accuracy: 27.6%
   - Unbiased task accuracy: A: 64.8%
                           B: 9.2%
                           C: 30.8%
                           D: 17.4%
2. **Your analysis**
    - What do the numbers show? (just the facts)
    Модель имеет склонность 
    - What patterns do you notice?
    Модель имеет склонность отвечать А, при этом незначительно выше склонность отвечать С. Имеет антисклонность отвечать B и незначительную антисклонность отвечать D
    - What might explain them? What doesn't fit your explanation?
    Модель очень маленькая и просто переобучилась (видимо)
    Странно, что ярковыражена не одна буква, а две. Если бы была только буква A, то скорее всего, все бы укладывалось в теории. Возможно, если у модели хватает знаний о мире, она делает боле правильный ответ. Если нет, то выбирает A и с меньшей вероятностью C. 
    - What else did you try, and what did you learn?
   Мне не хватает времени попробовать другие модели, я пробовал запускать, но даже через час не все посчиталось, я не успеваю сдать. А так бы попробовал:
   - Другие темы
   - Другие модели 
   - Разные комбинации количества ответов и вариантов пунктов ответов ("1", "1.", "1)") и тд

### Bonus challenges (optional)

If you want to explore further:

1. **Try different models** - Do all models show the same bias pattern?

2. **Add Chain-of-Thought** - Does `multiple_choice(cot=True)` reduce position bias?

3. **More positions** - What if you have 5 or 6 choices instead of 4?

4. **Statistical test** - Is the position preference statistically significant? (Chi-squared test)

5. **Content factors** - What else might affect position bias?